# Conformer-CTC training on Kaggle (2 x T4)

Trains the English speech recognition model on LibriSpeech within a fixed time budget.

**Before starting**, open *Settings* and select:
- *Accelerator*: **GPU T4 x2**
- *Internet*: **on** (requires a verified phone number)

Then choose *Run All*. Besides the training budget set below, the session spends some minutes downloading the data and evaluating the model. Those minutes also count against the GPU quota.

In [ ]:
TRAINING_MINUTES = 55
DATA = '/kaggle/temp/data'
LM = '/kaggle/temp/3-gram.pruned.3e-7.arpa.gz'
CHECKPOINTS = '/kaggle/working/checkpoints'

In [ ]:
!nvidia-smi -L
!df -h /kaggle/temp /kaggle/working
!git clone https://github.com/carlos-vi/ASR_experiments.git
%cd ASR_experiments
!pip install -q soundfile huggingface_hub pyarrow

## 1. Data

Downloads about 30 GB: LibriSpeech `train-clean-100`, `train-clean-360`, `dev-clean` and `test-clean`, plus the pretrained 3-gram language model from OpenSLR. Everything goes to `/kaggle/temp` because `/kaggle/working` has a limit on its size and number of files.

In [ ]:
!python -m asr_from_scratch.prepare_data --output {DATA} --splits train.100 train.360 validation test
!wget -q -O {LM} https://www.openslr.org/resources/11/3-gram.pruned.3e-7.arpa.gz

## 2. Training

The log reports throughput as a multiple of real time. If `nvidia-smi` shows plenty of free memory, `--batch-seconds` can go up; if the run stops with an out-of-memory error, lower it. If the session is interrupted, add `--resume` and run the cell again: training continues from the last evaluation, with the time budget intact.

In [ ]:
!torchrun --nproc_per_node=2 -m asr_from_scratch.train \
    --train {DATA}/train.100/manifest.jsonl {DATA}/train.360/manifest.jsonl \
    --val {DATA}/validation/manifest.jsonl \
    --output {CHECKPOINTS} --minutes {TRAINING_MINUTES}

## 3. Evaluation

Reports WER and CER on the test set, with greedy decoding and with beam search plus the language model. The language model weights are tuned on the validation set.

In [ ]:
!python -m asr_from_scratch.evaluate {CHECKPOINTS}/best.pt {DATA}/test/manifest.jsonl \
    --lm {LM} --tune-manifest {DATA}/validation/manifest.jsonl

## 4. Comparison with Whisper (optional)

Runs `small.en` on the CPU over a subset of the test clips, so it takes several minutes.

In [ ]:
!pip install -q faster-whisper
!python -m asr_from_scratch.evaluate {CHECKPOINTS}/best.pt {DATA}/test/manifest.jsonl \
    --max-clips 100 --whisper small.en

## 5. Download the model

`best.pt` is in the *Output* panel, under `checkpoints/`.